In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

In [2]:
# Generacion del dataset sintetico
print("Generando dataset sintético de 1,000 estudiantes de CONECTEDU...")
np.random.seed(42)
n_samples = 1000

Generando dataset sintético de 1,000 estudiantes de CONECTEDU...


In [3]:
# Variables basadas en la plataforma
perfil_vak = np.random.choice(['Visual', 'Auditivo', 'Kinestésico'], n_samples)
dias_inactivos = np.random.randint(0, 45, n_samples) # Días sin entrar a la plataforma
racha_actual = np.random.randint(0, 30, n_samples)   # Días seguidos estudiando (gamificación)
promedio_calificaciones = np.random.uniform(5.0, 10.0, n_samples)
tutorias_asistidas = np.random.randint(0, 10, n_samples) # Uso de la red de apoyo local

In [4]:
# Lógica de riesgo simulada: Más inactividad y menos tutorías/rachas = Mayor riesgo
# Agregamos algo de "ruido" aleatorio para que los algoritmos tengan que esforzarse
factor_riesgo = (dias_inactivos * 0.5) - (racha_actual * 0.3) - (promedio_calificaciones * 1.5) - (tutorias_asistidas * 2) + np.random.normal(0, 2, n_samples)

In [5]:
# El 30% de los alumnos con el factor de riesgo más alto serán clasificados como "1" (Alto Riesgo)
umbral = np.percentile(factor_riesgo, 70)
riesgo_desercion = np.where(factor_riesgo > umbral, 1, 0)

In [6]:
# Creación del DataFrame
df = pd.DataFrame({
    'perfil_vak': perfil_vak,
    'dias_inactivos': dias_inactivos,
    'racha_actual': racha_actual,
    'promedio_calificaciones': promedio_calificaciones,
    'tutorias_asistidas': tutorias_asistidas,
    'riesgo_desercion': riesgo_desercion
})

### Preparacion de los datos

In [7]:
X = df.drop('riesgo_desercion', axis=1)
y = df['riesgo_desercion']

In [8]:
# Dividimos en 80% entrenamiento y 20% prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [9]:
# Preprocesamiento: Escalar números y codificar el texto (VAK)
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['dias_inactivos', 'racha_actual', 'promedio_calificaciones', 'tutorias_asistidas']),
        ('cat', OneHotEncoder(), ['perfil_vak'])
    ])

### Entrenamiento de los 3 modelos

In [11]:
# Definimos los 3 algoritmos
modelos = {
    "Regresión Logística": LogisticRegression(),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Support Vector Machine (SVM)": SVC(probability=True, random_state=42)
}

resultados = []

print("\nEntrenando y evaluando modelos...\n" + "-"*40)

for nombre, modelo in modelos.items():
    # Creamos un pipeline que primero procesa los datos y luego entrena el modelo
    pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', modelo)])
    
    # Entrenamiento
    pipeline.fit(X_train, y_train)
    
    # Predicciones
    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1] # Probabilidades para el AUC-ROC
    
    # Cálculo estricto de las métricas que pide la rúbrica
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    
    resultados.append({'Modelo': nombre, 'Exactitud': acc, 'Precisión': prec, 'Recall': rec, 'F1-Score': f1, 'AUC-ROC': auc})


Entrenando y evaluando modelos...
----------------------------------------


### Resultados Finales

In [12]:
df_resultados = pd.DataFrame(resultados).round(4)
print(df_resultados.to_string(index=False))

                      Modelo  Exactitud  Precisión  Recall  F1-Score  AUC-ROC
         Regresión Logística       0.95     0.9800  0.8448    0.9074   0.9907
               Random Forest       0.92     0.9200  0.7931    0.8519   0.9843
Support Vector Machine (SVM)       0.94     0.9792  0.8103    0.8868   0.9875


### Exportar

In [13]:
import joblib

# 1. Recreamos el "tubo" (pipeline) especificando que usaremos Random Forest
modelo_ganador = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

# 2. Lo entrenamos con los datos que ya tenemos cargados en la memoria de Jupyter
modelo_ganador.fit(X_train, y_train)

# 3. ¡Exportamos el cerebro a un archivo físico!
archivo_exportado = 'modelo_riesgo_rf.joblib'
joblib.dump(modelo_ganador, archivo_exportado)

print(f"¡Éxito! El modelo se ha guardado como: {archivo_exportado}")
print("Revisa la carpeta de tu computadora donde está guardado este cuaderno de Jupyter.")

¡Éxito! El modelo se ha guardado como: modelo_riesgo_rf.joblib
Revisa la carpeta de tu computadora donde está guardado este cuaderno de Jupyter.
